# 🌸 Iris Flower Classification

A beginner-friendly ML project. We classify iris flowers into 3 species using sepal and petal measurements.

**Models:** Logistic Regression · Decision Tree · K-Nearest Neighbors

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
print('Libraries imported ✅')

## 1. Load & Explore Data

In [ ]:
df = pd.read_csv('../data/iris.csv')
print('Shape:', df.shape)
df.head(10)

In [ ]:
print(df['species'].value_counts())
print('Missing values:', df.isnull().sum().sum())
df.describe().round(2)

## 2. EDA — Distributions & Relationships

In [ ]:
feature_cols = [c for c in df.columns if c != 'species']
colors = {'setosa':'#FF6B6B','versicolor':'#4ECDC4','virginica':'#45B7D1'}

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle('Feature Distributions by Species', fontsize=15, fontweight='bold')
for ax, feat in zip(axes.flat, feature_cols):
    for sp, color in colors.items():
        ax.hist(df[df['species']==sp][feat], alpha=0.7, color=color, label=sp, bins=15, edgecolor='white')
    ax.set_title(feat.replace(' (cm)','').title(), fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
sns.pairplot(df, hue='species', palette=colors, diag_kind='kde')
plt.suptitle('Pairplot — All Features', y=1.02, fontsize=13, fontweight='bold')
plt.show()

In [ ]:
plt.figure(figsize=(7,5))
sns.heatmap(df[feature_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 3. Preprocessing

In [ ]:
X = df[feature_cols].values
y = df['species'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## 4. Train & Evaluate Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=200, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=4, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds  = model.predict(X_test)
    acc    = accuracy_score(y_test, preds)
    cv_acc = cross_val_score(model, X_train, y_train, cv=5).mean()
    results[name] = {'model':model,'accuracy':acc,'cv_accuracy':cv_acc,'preds':preds}
    print(f'{name:<25} Test: {acc:.4f}  CV: {cv_acc:.4f}')

In [ ]:
for name, r in results.items():
    print(f'\n=== {name} ===')
    print(classification_report(y_test, r['preds']))

## 5. Visualise Results

In [ ]:
names = list(results.keys())
accs  = [results[n]['accuracy']*100 for n in names]
cvs   = [results[n]['cv_accuracy']*100 for n in names]
x = np.arange(len(names)); width = 0.35

fig, ax = plt.subplots(figsize=(9,5))
b1 = ax.bar(x-width/2, accs, width, label='Test', color='#FF6B6B', alpha=0.9)
b2 = ax.bar(x+width/2, cvs,  width, label='CV',   color='#4ECDC4', alpha=0.9)
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylim(80,105); ax.set_ylabel('Accuracy (%)')
ax.set_title('Model Comparison', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4, f'{bar.get_height():.1f}%', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(16,5))
fig.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
for ax,(name,r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['preds'])
    ConfusionMatrixDisplay(cm, display_labels=['setosa','versicolor','virginica']).plot(ax=ax,colorbar=False,cmap='Blues')
    ax.set_title(name, fontweight='bold', fontsize=10)
plt.tight_layout(); plt.show()

## 6. Predict a New Flower

In [ ]:
# [sepal_length, sepal_width, petal_length, petal_width] in cm
new_flower = np.array([[5.1, 3.5, 1.4, 0.2]])
new_scaled = scaler.transform(new_flower)

best_name  = max(results, key=lambda n: results[n]['accuracy'])
best_model = results[best_name]['model']

pred  = best_model.predict(new_scaled)[0]
proba = best_model.predict_proba(new_scaled)[0]

print(f'Predicted: {pred.upper()}')
for cls, p in zip(best_model.classes_, proba):
    print(f'  {cls:<12} {"█"*int(p*30)} {p*100:.1f}%')

## Key Takeaways

- **Petal features** are more informative than sepal features for distinguishing species.
- **Setosa** is easily separated; Versicolor and Virginica overlap slightly.
- All three models reach **~93% accuracy**.
- Feature scaling matters for Logistic Regression and KNN.

### Next steps
- Tune hyperparameters (`n_neighbors`, `max_depth`)
- Try Random Forest or SVM
- Use `GridSearchCV` for automated search